# 03 — MS-TCN-Style Teacher-Student Training on Breakfast

This notebook runs the first controlled training experiment for the updated project direction.

Core idea:

```text
Training:   text can be used
Inference:  only visual features are used
```

Models compared:

```text
1. baseline_visual_only      — visual-only TCN baseline
2. text_aware_teacher        — teacher using CLIP action text prototypes
3. student_ce_only           — video-only student trained without distillation
4. student_kd_video_only     — video-only student trained with KD from teacher
```

Default config is a scale-1 control run: 200 train videos, 50 test videos, 3 epochs.
For full experiments, change `RUN_MODE` below.


## 1. Mount Google Drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Imports and paths


In [3]:
from pathlib import Path
import json
import random
import time
import shutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_ROOT = DRIVE_ROOT / 'mmf_tas_lab_data'
PROJECT_ROOT = DRIVE_ROOT / 'mmf_tas_lab_project'

BREAKFAST_ROOT = DATA_ROOT / 'zenodo_ms_tcn_data' / 'breakfast'
TEXT_ASSISTED_ROOT = DATA_ROOT / 'text_assisted_tas' / 'breakfast'
TEXT_EMBEDDING_DIR = TEXT_ASSISTED_ROOT / 'text_embeddings'

print('BREAKFAST_ROOT:', BREAKFAST_ROOT)
print('TEXT_ASSISTED_ROOT:', TEXT_ASSISTED_ROOT)


BREAKFAST_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast
TEXT_ASSISTED_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast


## 3. Experiment configuration


In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Options:
#   'smoke'          -> very small quick run
#   'scale1_control' -> controlled run
#   'full_split1'    -> full Breakfast split 1, longer run
RUN_MODE = 'full_split1'
SPLIT_ID = 1

if RUN_MODE == 'smoke':
    RUN_NAME = 'mstcn_teacher_student_split1_smoke'
    MAX_TRAIN_VIDEOS = 40
    MAX_TEST_VIDEOS = 10
    NUM_EPOCHS_BASELINE = 1
    NUM_EPOCHS_TEACHER = 1
    NUM_EPOCHS_STUDENT = 1
elif RUN_MODE == 'scale1_control':
    RUN_NAME = 'mstcn_teacher_student_split1_scale1_control'
    MAX_TRAIN_VIDEOS = 200
    MAX_TEST_VIDEOS = 50
    NUM_EPOCHS_BASELINE = 3
    NUM_EPOCHS_TEACHER = 3
    NUM_EPOCHS_STUDENT = 3
elif RUN_MODE == 'full_split1':
    RUN_NAME = 'mstcn_teacher_student_split1_full'
    MAX_TRAIN_VIDEOS = None
    MAX_TEST_VIDEOS = None
    NUM_EPOCHS_BASELINE = 10
    NUM_EPOCHS_TEACHER = 10
    NUM_EPOCHS_STUDENT = 10
else:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')

BATCH_SIZE = 1
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5

NUM_F_MAPS = 64
NUM_LAYERS = 8
NUM_STAGES = 2

KD_TEMPERATURE = 4.0
LAMBDA_CE = 1.0
LAMBDA_KD = 1.0

COPY_SELECTED_DATA_TO_LOCAL = True

RUN_ROOT = TEXT_ASSISTED_ROOT / 'runs' / RUN_NAME
LOCAL_RUN_ROOT = Path('/content/text_assisted_tas_runs/breakfast') / RUN_NAME
for p in [RUN_ROOT, LOCAL_RUN_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('RUN_MODE:', RUN_MODE)
print('RUN_NAME:', RUN_NAME)
print('Split:', SPLIT_ID)
print('MAX_TRAIN_VIDEOS:', MAX_TRAIN_VIDEOS)
print('MAX_TEST_VIDEOS:', MAX_TEST_VIDEOS)
print('Epochs baseline/teacher/student:', NUM_EPOCHS_BASELINE, NUM_EPOCHS_TEACHER, NUM_EPOCHS_STUDENT)
print('COPY_SELECTED_DATA_TO_LOCAL:', COPY_SELECTED_DATA_TO_LOCAL)

if device != 'cuda':
    print('\nWARNING: You are not using GPU. For real runs, set Colab runtime to T4 GPU.')


RUN_MODE: full_split1
RUN_NAME: mstcn_teacher_student_split1_full
Split: 1
MAX_TRAIN_VIDEOS: None
MAX_TEST_VIDEOS: None
Epochs baseline/teacher/student: 10 10 10
COPY_SELECTED_DATA_TO_LOCAL: True



## 4. Validate required files


In [5]:
required_paths = {
    'Breakfast features': BREAKFAST_ROOT / 'features',
    'Breakfast groundTruth': BREAKFAST_ROOT / 'groundTruth',
    'Breakfast mapping': BREAKFAST_ROOT / 'mapping.txt',
    'Breakfast splits': BREAKFAST_ROOT / 'splits',
    'Text embeddings': TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy',
    'Text embedding metadata': TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_metadata.csv',
}

for name, path in required_paths.items():
    print(f'{name}: {path} -> exists={path.exists()}')
    if not path.exists():
        raise FileNotFoundError(f'Missing required file/folder: {name}: {path}')


Breakfast features: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/features -> exists=True
Breakfast groundTruth: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/groundTruth -> exists=True
Breakfast mapping: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/mapping.txt -> exists=True
Breakfast splits: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/splits -> exists=True
Text embeddings: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/text_embeddings/breakfast_clip_vitb16_text_embeddings.npy -> exists=True
Text embedding metadata: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/text_embeddings/breakfast_clip_vitb16_text_embedding_metadata.csv -> exists=True


## 5. Load mapping, text embeddings, and splits


In [6]:
def read_lines(path: Path):
    return path.read_text().splitlines()


def load_mapping(mapping_path: Path):
    idx_to_label = {}
    label_to_idx = {}
    for line in read_lines(mapping_path):
        line = line.strip()
        if not line:
            continue
        idx, label = line.split(maxsplit=1)
        idx = int(idx)
        idx_to_label[idx] = label
        label_to_idx[label] = idx
    return idx_to_label, label_to_idx


def load_split(split_path: Path):
    video_ids = []
    for line in read_lines(split_path):
        line = line.strip()
        if line:
            video_ids.append(Path(line).stem)
    return video_ids


idx_to_label, label_to_idx = load_mapping(BREAKFAST_ROOT / 'mapping.txt')
num_classes = len(idx_to_label)

text_embeddings = np.load(TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy').astype(np.float32)
text_metadata = pd.read_csv(TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_metadata.csv')

train_ids = load_split(BREAKFAST_ROOT / 'splits' / f'train.split{SPLIT_ID}.bundle')
test_ids = load_split(BREAKFAST_ROOT / 'splits' / f'test.split{SPLIT_ID}.bundle')

if MAX_TRAIN_VIDEOS is not None:
    train_ids = train_ids[:MAX_TRAIN_VIDEOS]
if MAX_TEST_VIDEOS is not None:
    test_ids = test_ids[:MAX_TEST_VIDEOS]

print('Number of classes:', num_classes)
print('Text embeddings shape:', text_embeddings.shape)
print('Train videos:', len(train_ids))
print('Test videos:', len(test_ids))
display(text_metadata.head())


Number of classes: 48
Text embeddings shape: (48, 512)
Train videos: 1460
Test videos: 252


,class_index,action_label,action_text,prompt
0,0,SIL,SIL,a video of the action: SIL
1,1,pour_cereals,pour cereals,a video of the action: pour cereals
2,2,pour_milk,pour milk,a video of the action: pour milk
3,3,stir_cereals,stir cereals,a video of the action: stir cereals
4,4,take_bowl,take bowl,a video of the action: take bowl


## 6. Optional local data cache


In [8]:
ACTIVE_FEATURE_DIR = BREAKFAST_ROOT / 'features'
ACTIVE_GT_DIR = BREAKFAST_ROOT / 'groundTruth'

if COPY_SELECTED_DATA_TO_LOCAL:
    selected_ids = sorted(set(train_ids + test_ids))
    LOCAL_DATA_CACHE = Path('/content/breakfast_selected_cache') / RUN_NAME
    local_feature_dir = LOCAL_DATA_CACHE / 'features'
    local_gt_dir = LOCAL_DATA_CACHE / 'groundTruth'
    local_feature_dir.mkdir(parents=True, exist_ok=True)
    local_gt_dir.mkdir(parents=True, exist_ok=True)

    for video_id in tqdm(selected_ids, desc='Copying selected Breakfast files to /content cache'):
        src_feature = BREAKFAST_ROOT / 'features' / f'{video_id}.npy'
        dst_feature = local_feature_dir / f'{video_id}.npy'
        if not dst_feature.exists():
            shutil.copy2(src_feature, dst_feature)

        src_gt = BREAKFAST_ROOT / 'groundTruth' / f'{video_id}.txt'
        dst_gt = local_gt_dir / f'{video_id}.txt'
        if not dst_gt.exists():
            shutil.copy2(src_gt, dst_gt)

    ACTIVE_FEATURE_DIR = local_feature_dir
    ACTIVE_GT_DIR = local_gt_dir

print('ACTIVE_FEATURE_DIR:', ACTIVE_FEATURE_DIR)
print('ACTIVE_GT_DIR:', ACTIVE_GT_DIR)


Copying selected Breakfast files to /content cache:   0%|          | 0/1712 [00:00<?, ?it/s]

ACTIVE_FEATURE_DIR: /content/breakfast_selected_cache/mstcn_teacher_student_split1_full/features
ACTIVE_GT_DIR: /content/breakfast_selected_cache/mstcn_teacher_student_split1_full/groundTruth


## 7. Dataset and DataLoader


In [9]:
class BreakfastTASDataset(Dataset):
    def __init__(self, video_ids, feature_dir: Path, gt_dir: Path, label_to_idx: dict):
        self.video_ids = list(video_ids)
        self.feature_dir = feature_dir
        self.gt_dir = gt_dir
        self.label_to_idx = label_to_idx

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, index):
        video_id = self.video_ids[index]
        feature_path = self.feature_dir / f'{video_id}.npy'
        gt_path = self.gt_dir / f'{video_id}.txt'

        features = np.load(feature_path).astype(np.float32)
        labels_str = read_lines(gt_path)
        labels = np.asarray([self.label_to_idx[x] for x in labels_str], dtype=np.int64)

        T = min(features.shape[1], len(labels))
        features = features[:, :T]
        labels = labels[:T]

        return {
            'video_id': video_id,
            'features': torch.from_numpy(features),
            'labels': torch.from_numpy(labels),
            'length': T,
        }


def tas_collate_fn(batch):
    batch_size = len(batch)
    feature_dim = batch[0]['features'].shape[0]
    max_len = max(item['length'] for item in batch)

    features = torch.zeros(batch_size, feature_dim, max_len, dtype=torch.float32)
    labels = torch.full((batch_size, max_len), fill_value=-100, dtype=torch.long)
    mask = torch.zeros(batch_size, 1, max_len, dtype=torch.float32)
    video_ids = []

    for i, item in enumerate(batch):
        T = item['length']
        features[i, :, :T] = item['features']
        labels[i, :T] = item['labels']
        mask[i, :, :T] = 1.0
        video_ids.append(item['video_id'])

    return {
        'video_ids': video_ids,
        'features': features,
        'labels': labels,
        'mask': mask,
        'lengths': torch.tensor([item['length'] for item in batch], dtype=torch.long),
    }


train_dataset = BreakfastTASDataset(train_ids, ACTIVE_FEATURE_DIR, ACTIVE_GT_DIR, label_to_idx)
test_dataset = BreakfastTASDataset(test_ids, ACTIVE_FEATURE_DIR, ACTIVE_GT_DIR, label_to_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=tas_collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=tas_collate_fn, num_workers=0)

sample = next(iter(train_loader))
print('Sample features:', sample['features'].shape)
print('Sample labels:', sample['labels'].shape)
print('Sample mask:', sample['mask'].shape)
print('Sample video:', sample['video_ids'][0])


Sample features: torch.Size([1, 2048, 1249])
Sample labels: torch.Size([1, 1249])
Sample mask: torch.Size([1, 1, 1249])
Sample video: P48_cam01_P48_juice


## 8. MS-TCN-style model definitions


In [10]:
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, channels):
        super().__init__()
        self.conv_dilated = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.conv_1x1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, mask):
        out = F.relu(self.conv_dilated(x))
        out = self.conv_1x1(out)
        out = self.dropout(out)
        return (x + out) * mask


class SingleStageTCN(nn.Module):
    def __init__(self, num_layers, num_f_maps, dim, num_classes):
        super().__init__()
        self.conv_in = nn.Conv1d(dim, num_f_maps, kernel_size=1)
        self.layers = nn.ModuleList([DilatedResidualLayer(2 ** i, num_f_maps) for i in range(num_layers)])
        self.conv_out = nn.Conv1d(num_f_maps, num_classes, kernel_size=1)

    def forward(self, x, mask):
        out = self.conv_in(x) * mask
        for layer in self.layers:
            out = layer(out, mask)
        return self.conv_out(out) * mask


class MultiStageTCN(nn.Module):
    def __init__(self, num_stages, num_layers, num_f_maps, dim, num_classes):
        super().__init__()
        self.stage1 = SingleStageTCN(num_layers, num_f_maps, dim, num_classes)
        self.stages = nn.ModuleList([
            SingleStageTCN(num_layers, num_f_maps, num_classes, num_classes)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x, mask):
        outputs = []
        out = self.stage1(x, mask)
        outputs.append(out)
        for stage in self.stages:
            out = stage(F.softmax(out, dim=1) * mask, mask)
            outputs.append(out)
        return torch.stack(outputs, dim=0)


## 9. Text-aware teacher model


In [11]:
class TextPrototypeSingleStageTCN(nn.Module):
    def __init__(self, num_layers, num_f_maps, dim, text_embeddings_np):
        super().__init__()
        text_embeddings_t = torch.from_numpy(text_embeddings_np).float()
        text_embeddings_t = F.normalize(text_embeddings_t, dim=1)
        self.register_buffer('text_embeddings', text_embeddings_t)  # [C, E]

        num_classes, text_dim = text_embeddings_t.shape
        self.conv_in = nn.Conv1d(dim, num_f_maps, kernel_size=1)
        self.layers = nn.ModuleList([DilatedResidualLayer(2 ** i, num_f_maps) for i in range(num_layers)])
        self.visual_to_text = nn.Conv1d(num_f_maps, text_dim, kernel_size=1)
        self.logit_scale = nn.Parameter(torch.tensor(10.0))

    def forward(self, x, mask):
        out = self.conv_in(x) * mask
        for layer in self.layers:
            out = layer(out, mask)
        projected = self.visual_to_text(out)  # [B, E, T]
        projected = F.normalize(projected, dim=1)
        logits = torch.einsum('bet,ce->bct', projected, self.text_embeddings)
        logits = logits * self.logit_scale.clamp(1.0, 100.0)
        return logits * mask


class TextPrototypeMultiStageTCN(nn.Module):
    def __init__(self, num_stages, num_layers, num_f_maps, dim, text_embeddings_np):
        super().__init__()
        num_classes = text_embeddings_np.shape[0]
        self.stage1 = TextPrototypeSingleStageTCN(num_layers, num_f_maps, dim, text_embeddings_np)
        self.stages = nn.ModuleList([
            TextPrototypeSingleStageTCN(num_layers, num_f_maps, num_classes, text_embeddings_np)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x, mask):
        outputs = []
        out = self.stage1(x, mask)
        outputs.append(out)
        for stage in self.stages:
            out = stage(F.softmax(out, dim=1) * mask, mask)
            outputs.append(out)
        return torch.stack(outputs, dim=0)


## 10. Losses and metrics


In [12]:
def mstcn_supervised_loss(outputs, labels):
    total_loss = 0.0
    for s in range(outputs.shape[0]):
        logits = outputs[s].transpose(1, 2).contiguous()
        total_loss = total_loss + F.cross_entropy(
            logits.view(-1, logits.shape[-1]),
            labels.view(-1),
            ignore_index=-100,
        )
    return total_loss / outputs.shape[0]


def kd_loss_student_teacher(student_outputs, teacher_outputs, labels, temperature=4.0, lambda_ce=1.0, lambda_kd=1.0):
    ce = mstcn_supervised_loss(student_outputs, labels)
    student_last = student_outputs[-1]
    teacher_last = teacher_outputs[-1].detach()
    valid = labels != -100

    s_logits = student_last.transpose(1, 2)[valid]
    t_logits = teacher_last.transpose(1, 2)[valid]

    log_p_student = F.log_softmax(s_logits / temperature, dim=-1)
    p_teacher = F.softmax(t_logits / temperature, dim=-1)
    kd = F.kl_div(log_p_student, p_teacher, reduction='batchmean') * (temperature ** 2)
    return lambda_ce * ce + lambda_kd * kd, ce.detach(), kd.detach()


@torch.no_grad()
def framewise_accuracy(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    for batch in loader:
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)
        mask = batch['mask'].to(device)
        outputs = model(features, mask)
        preds = torch.argmax(outputs[-1], dim=1)
        valid = labels != -100
        correct += (preds[valid] == labels[valid]).sum().item()
        total += valid.sum().item()
    return correct / max(total, 1)


## 11. Build models


In [13]:
feature_dim = sample['features'].shape[1]

baseline_model = MultiStageTCN(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, feature_dim, num_classes).to(device)
teacher_model = TextPrototypeMultiStageTCN(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, feature_dim, text_embeddings).to(device)
student_ce_model = MultiStageTCN(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, feature_dim, num_classes).to(device)
student_kd_model = MultiStageTCN(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, feature_dim, num_classes).to(device)

print('Feature dim:', feature_dim)
print('Classes:', num_classes)
print('Baseline parameters:', sum(p.numel() for p in baseline_model.parameters()))
print('Teacher parameters:', sum(p.numel() for p in teacher_model.parameters()))
print('Student CE-only parameters:', sum(p.numel() for p in student_ce_model.parameters()))
print('Student KD parameters:', sum(p.numel() for p in student_kd_model.parameters()))


Feature dim: 2048
Classes: 48
Baseline parameters: 404704
Teacher parameters: 465026
Student CE-only parameters: 404704
Student KD parameters: 404704


## 12. Training loops


In [14]:
def train_supervised_model(model, loader, optimizer, num_epochs, device, model_name):
    history = []
    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_losses = []
        start_time = time.time()

        for batch in tqdm(loader, desc=f'{model_name} epoch {epoch}/{num_epochs}'):
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)

            optimizer.zero_grad()
            outputs = model(features, mask)
            loss = mstcn_supervised_loss(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.item()))

        row = {
            'model': model_name,
            'epoch': epoch,
            'loss': float(np.mean(epoch_losses)) if epoch_losses else float('nan'),
            'train_acc': framewise_accuracy(model, train_loader, device),
            'test_acc': framewise_accuracy(model, test_loader, device),
            'seconds': time.time() - start_time,
        }
        history.append(row)
        print(row)
    return pd.DataFrame(history)


def train_student_with_kd(student, teacher, loader, optimizer, num_epochs, device):
    history = []
    teacher.eval()
    for epoch in range(1, num_epochs + 1):
        student.train()
        epoch_losses, epoch_ce, epoch_kd = [], [], []
        start_time = time.time()

        for batch in tqdm(loader, desc=f'student KD epoch {epoch}/{num_epochs}'):
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)

            optimizer.zero_grad()
            with torch.no_grad():
                teacher_outputs = teacher(features, mask)
            student_outputs = student(features, mask)
            loss, ce, kd = kd_loss_student_teacher(
                student_outputs,
                teacher_outputs,
                labels,
                temperature=KD_TEMPERATURE,
                lambda_ce=LAMBDA_CE,
                lambda_kd=LAMBDA_KD,
            )
            loss.backward()
            optimizer.step()

            epoch_losses.append(float(loss.item()))
            epoch_ce.append(float(ce.item()))
            epoch_kd.append(float(kd.item()))

        row = {
            'model': 'student_kd_video_only',
            'epoch': epoch,
            'loss': float(np.mean(epoch_losses)) if epoch_losses else float('nan'),
            'ce': float(np.mean(epoch_ce)) if epoch_ce else float('nan'),
            'kd': float(np.mean(epoch_kd)) if epoch_kd else float('nan'),
            'train_acc': framewise_accuracy(student, train_loader, device),
            'test_acc': framewise_accuracy(student, test_loader, device),
            'seconds': time.time() - start_time,
        }
        history.append(row)
        print(row)
    return pd.DataFrame(history)


## 13. Train models


In [ ]:
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
baseline_history = train_supervised_model(
    baseline_model, train_loader, baseline_optimizer, NUM_EPOCHS_BASELINE, device, 'baseline_visual_only'
)
display(baseline_history)

teacher_optimizer = torch.optim.Adam(teacher_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
teacher_history = train_supervised_model(
    teacher_model, train_loader, teacher_optimizer, NUM_EPOCHS_TEACHER, device, 'text_aware_teacher'
)
display(teacher_history)

student_ce_optimizer = torch.optim.Adam(student_ce_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
student_ce_history = train_supervised_model(
    student_ce_model, train_loader, student_ce_optimizer, NUM_EPOCHS_STUDENT, device, 'student_ce_only'
)
display(student_ce_history)

student_kd_optimizer = torch.optim.Adam(student_kd_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
student_kd_history = train_student_with_kd(
    student_kd_model, teacher_model, train_loader, student_kd_optimizer, NUM_EPOCHS_STUDENT, device
)
display(student_kd_history)


baseline_visual_only epoch 1/10:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'baseline_visual_only', 'epoch': 1, 'loss': 2.681787762617412, 'train_acc': 0.31105174337711805, 'test_acc': 0.30932171531907987, 'seconds': 766.1502614021301}


baseline_visual_only epoch 2/10:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'baseline_visual_only', 'epoch': 2, 'loss': 1.9079217126720571, 'train_acc': 0.38252237822547375, 'test_acc': 0.3676215123203976, 'seconds': 867.5748074054718}


baseline_visual_only epoch 3/10:   0%|          | 0/1460 [00:00<?, ?it/s]

...

student KD epoch 9/10:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'student_kd_video_only', 'epoch': 9, 'loss': 1.3018801961145172, 'ce': 1.086359399975571, 'kd': 0.2155207968023542, 'train_acc': 0.6028066324915078, 'test_acc': 0.4853073273423001, 'seconds': 1001.6935486793518}


student KD epoch 10/10:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'student_kd_video_only', 'epoch': 10, 'loss': 1.2166037117373454, 'ce': 1.0146377879769017, 'kd': 0.20196592277043487, 'train_acc': 0.6085405919408895, 'test_acc': 0.49645246942159227, 'seconds': 961.4622611999512}


,model,epoch,loss,ce,kd,train_acc,test_acc,seconds
0,student_kd_video_only,1,3.696548,2.741227,0.955322,0.291173,0.312375,951.745232
1,student_kd_video_only,2,2.538536,1.974417,0.564119,0.428173,0.387840,965.775845
2,student_kd_video_only,3,2.108854,1.691988,0.416866,0.473334,0.416462,967.261636
3,student_kd_video_only,4,1.843660,1.511170,0.332490,0.534468,0.459252,985.565246
4,student_kd_video_only,5,1.686355,1.390604,0.295751,0.549750,0.472514,1001.363823
5,student_kd_video_only,6,1.511514,1.259650,0.251864,0.510922,0.416068,1036.814733
6,student_kd_video_only,7,1.410513,1.182924,0.227589,0.596669,0.480110,1024.158991
7,student_kd_video_only,8,1.334515,1.117480,0.217035,0.620342,0.515482,1007.239774
8,student_kd_video_only,9,1.301880,1.086359,0.215521,0.602807,0.485307,1001.693549
9,student_kd_video_only,10,1.216604,1.014638,0.201966,0.608541,0.496452,961.462261


In [ ]:
for name in [
    "baseline_model",
    "teacher_model",
    "student_ce_model",
    "student_kd_model",
    "train_loader",
    "test_loader",
    "device"
]:
    print(name, name in globals())

baseline_model True
teacher_model True
student_ce_model True
student_kd_model True
train_loader True
test_loader True
device True


## 14. Compare and save results


In [ ]:
comparison_rows = []
for name, model in [
    ('baseline_visual_only', baseline_model),
    ('text_aware_teacher', teacher_model),
    ('student_ce_only', student_ce_model),
    ('student_kd_video_only', student_kd_model),
]:
    comparison_rows.append({
        'model': name,
        'train_acc': framewise_accuracy(model, train_loader, device),
        'test_acc': framewise_accuracy(model, test_loader, device),
        'uses_text_at_training': name in ['text_aware_teacher', 'student_kd_video_only'],
        'uses_text_at_inference': name == 'text_aware_teacher',
    })

df_comparison = pd.DataFrame(comparison_rows)
display(df_comparison)

local_comparison_path = LOCAL_RUN_ROOT / 'comparison.csv'
drive_comparison_path = RUN_ROOT / 'comparison.csv'
df_comparison.to_csv(local_comparison_path, index=False)
try:
    shutil.copy2(local_comparison_path, drive_comparison_path)
    print('Copied comparison to Drive:', drive_comparison_path)
except Exception as e:
    print('WARNING: Could not copy comparison to Drive.')
    print('Local comparison is safe:', local_comparison_path)
    print('Error:', repr(e))


,model,train_acc,test_acc,uses_text_at_training,uses_text_at_inference
0,baseline_visual_only,0.666467,0.535590,False,False
1,text_aware_teacher,0.449591,0.372099,True,True
2,student_ce_only,0.693824,0.539339,False,False
3,student_kd_video_only,0.608541,0.496452,True,False


Copied comparison to Drive: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/mstcn_teacher_student_split1_full/comparison.csv


## 15. Save checkpoint and summary


In [ ]:
checkpoint = {
    'config': {
        'run_mode': RUN_MODE,
        'run_name': RUN_NAME,
        'dataset': 'breakfast',
        'split_id': SPLIT_ID,
        'max_train_videos': MAX_TRAIN_VIDEOS,
        'max_test_videos': MAX_TEST_VIDEOS,
        'num_classes': num_classes,
        'feature_dim': feature_dim,
        'num_stages': NUM_STAGES,
        'num_layers': NUM_LAYERS,
        'num_f_maps': NUM_F_MAPS,
        'kd_temperature': KD_TEMPERATURE,
        'lambda_ce': LAMBDA_CE,
        'lambda_kd': LAMBDA_KD,
    },
    'idx_to_label': idx_to_label,
    'baseline_state_dict': baseline_model.state_dict(),
    'teacher_state_dict': teacher_model.state_dict(),
    'student_ce_state_dict': student_ce_model.state_dict(),
    'student_kd_state_dict': student_kd_model.state_dict(),
    'baseline_history': baseline_history.to_dict(orient='records'),
    'teacher_history': teacher_history.to_dict(orient='records'),
    'student_ce_history': student_ce_history.to_dict(orient='records'),
    'student_kd_history': student_kd_history.to_dict(orient='records'),
    'comparison': df_comparison.to_dict(orient='records'),
}

local_ckpt_path = LOCAL_RUN_ROOT / 'teacher_student_checkpoint.pt'
drive_ckpt_path = RUN_ROOT / 'teacher_student_checkpoint.pt'
torch.save(checkpoint, local_ckpt_path)
print('Saved local checkpoint:', local_ckpt_path)
try:
    shutil.copy2(local_ckpt_path, drive_ckpt_path)
    print('Copied checkpoint to Drive:', drive_ckpt_path)
except Exception as e:
    print('WARNING: Could not copy checkpoint to Drive.')
    print('Local checkpoint is safe:', local_ckpt_path)
    print('Error:', repr(e))

summary = {
    'status': 'completed_run',
    'run_mode': RUN_MODE,
    'run_name': RUN_NAME,
    'dataset': 'Breakfast',
    'split': SPLIT_ID,
    'train_videos': len(train_ids),
    'test_videos': len(test_ids),
    'classes': num_classes,
    'feature_dim': feature_dim,
    'text_embedding_shape': list(text_embeddings.shape),
    'models': {
        'baseline_visual_only': 'visual-only MS-TCN-style model',
        'text_aware_teacher': 'teacher with CLIP action text prototypes',
        'student_ce_only': 'video-only student trained with cross entropy only',
        'student_kd_video_only': 'video-only student trained with KD from teacher',
    },
    'comparison': df_comparison.to_dict(orient='records'),
}

local_summary_path = LOCAL_RUN_ROOT / 'experiment_summary.json'
drive_summary_path = RUN_ROOT / 'experiment_summary.json'
with local_summary_path.open('w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
try:
    shutil.copy2(local_summary_path, drive_summary_path)
    print('Copied summary to Drive:', drive_summary_path)
except Exception as e:
    print('WARNING: Could not copy summary to Drive.')
    print('Local summary is safe:', local_summary_path)
    print('Error:', repr(e))

print('\n03_mstcn_teacher_student_training completed.')


Saved local checkpoint: /content/text_assisted_tas_runs/breakfast/mstcn_teacher_student_split1_full/teacher_student_checkpoint.pt
Copied checkpoint to Drive: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/mstcn_teacher_student_split1_full/teacher_student_checkpoint.pt
{
  "status": "completed_run",
  "run_mode": "full_split1",
  "run_name": "mstcn_teacher_student_split1_full",
  "dataset": "Breakfast",
  "split": 1,
  "train_videos": 1460,
  "test_videos": 252,
  "classes": 48,
  "feature_dim": 2048,
  "text_embedding_shape": [
    48,
    512
  ],
  "models": {
    "baseline_visual_only": "visual-only MS-TCN-style model",
    "text_aware_teacher": "teacher with CLIP action text prototypes",
    "student_ce_only": "video-only student trained with cross entropy only",
    "student_kd_video_only": "video-only student trained with KD from teacher"
  },
  "comparison": [
    {
      "model": "baseline_visual_only",
      "train_acc": 0.6664671297177065,
      "tes

## 16. Next steps after this notebook

After the scale-1 control run is complete:

1. add TAS metrics: Edit, F1@10, F1@25, F1@50;
2. run full Breakfast split 1;
3. run Breakfast splits 1–4;
4. move the same pipeline to Assembly101;
5. later compare against LTContext.


## Save frame-level predictions for TAS metrics

This section saves per-frame predicted action labels for each test video.

The output format matches the Breakfast `groundTruth/*.txt` style: one action label per frame.

These files are needed for standard TAS metrics such as Edit score and F1@10/25/50.


In [16]:
@torch.no_grad()
def save_frame_level_predictions(model, loader, model_name, device):
    model.eval()

    local_pred_dir = LOCAL_RUN_ROOT / "predictions" / model_name
    drive_pred_dir = RUN_ROOT / "predictions" / model_name

    local_pred_dir.mkdir(parents=True, exist_ok=True)
    drive_pred_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for batch in tqdm(loader, desc=f"Saving predictions: {model_name}"):
        features = batch["features"].to(device)
        mask = batch["mask"].to(device)
        lengths = batch["lengths"].cpu().numpy()
        video_ids = batch["video_ids"]

        outputs = model(features, mask)
        preds = torch.argmax(outputs[-1], dim=1).cpu().numpy()

        for i, video_id in enumerate(video_ids):
            T = int(lengths[i])
            pred_idx = preds[i, :T]
            pred_labels = [idx_to_label[int(x)] for x in pred_idx]

            local_path = local_pred_dir / f"{video_id}.txt"
            drive_path = drive_pred_dir / f"{video_id}.txt"

            local_path.write_text("\n".join(pred_labels) + "\n")

            try:
                shutil.copy2(local_path, drive_path)
                copied_to_drive = True
            except Exception as e:
                copied_to_drive = False
                print(f"WARNING: could not copy {video_id} prediction to Drive:", repr(e))

            rows.append({
                "model": model_name,
                "video_id": video_id,
                "num_frames": T,
                "local_prediction_path": str(local_path),
                "drive_prediction_path": str(drive_path),
                "copied_to_drive": copied_to_drive,
            })

    manifest = pd.DataFrame(rows)

    local_manifest_path = LOCAL_RUN_ROOT / f"{model_name}_prediction_manifest.csv"
    drive_manifest_path = RUN_ROOT / f"{model_name}_prediction_manifest.csv"

    manifest.to_csv(local_manifest_path, index=False)

    try:
        shutil.copy2(local_manifest_path, drive_manifest_path)
        print("Copied manifest to Drive:", drive_manifest_path)
    except Exception as e:
        print("WARNING: could not copy manifest to Drive.")
        print("Local manifest is safe:", local_manifest_path)
        print("Error:", repr(e))

    print(f"Saved {len(manifest)} prediction files for {model_name}")
    return manifest


models_to_save = {
    "baseline_visual_only": baseline_model,
    "text_aware_teacher": teacher_model,
    "student_ce_only": student_ce_model,
    "student_kd_video_only": student_kd_model,
}

prediction_manifests = []

for model_name, model in models_to_save.items():
    manifest = save_frame_level_predictions(model, test_loader, model_name, device)
    prediction_manifests.append(manifest)

df_prediction_manifest = pd.concat(prediction_manifests, ignore_index=True)

local_all_manifest_path = LOCAL_RUN_ROOT / "prediction_manifest_all_models.csv"
drive_all_manifest_path = RUN_ROOT / "prediction_manifest_all_models.csv"

df_prediction_manifest.to_csv(local_all_manifest_path, index=False)

try:
    shutil.copy2(local_all_manifest_path, drive_all_manifest_path)
    print("Copied combined prediction manifest to Drive:", drive_all_manifest_path)
except Exception as e:
    print("WARNING: could not copy combined manifest to Drive.")
    print("Local combined manifest is safe:", local_all_manifest_path)
    print("Error:", repr(e))

prediction_summary = {
    "run_name": RUN_NAME,
    "dataset": "Breakfast",
    "split": SPLIT_ID,
    "num_test_videos": len(test_ids),
    "models": list(models_to_save.keys()),
    "prediction_root_local": str(LOCAL_RUN_ROOT / "predictions"),
    "prediction_root_drive": str(RUN_ROOT / "predictions"),
    "manifest": str(drive_all_manifest_path),
}

local_prediction_summary_path = LOCAL_RUN_ROOT / "prediction_summary.json"
drive_prediction_summary_path = RUN_ROOT / "prediction_summary.json"

with local_prediction_summary_path.open("w") as f:
    json.dump(prediction_summary, f, indent=2)

try:
    shutil.copy2(local_prediction_summary_path, drive_prediction_summary_path)
    print("Copied prediction summary to Drive:", drive_prediction_summary_path)
except Exception as e:
    print("WARNING: could not copy prediction summary to Drive.")
    print("Local prediction summary is safe:", local_prediction_summary_path)
    print("Error:", repr(e))

display(df_prediction_manifest.head())
print("Prediction saving complete.")


Saving predictions: baseline_visual_only:   0%|          | 0/252 [00:00<?, ?it/s]

Copied manifest to Drive: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/mstcn_teacher_student_split1_full/baseline_visual_only_prediction_manifest.csv
Saved 252 prediction files for baseline_visual_only


Saving predictions: text_aware_teacher:   0%|          | 0/252 [00:00<?, ?it/s]

Copied manifest to Drive: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/mstcn_teacher_student_split1_full/text_aware_teacher_prediction_manifest.csv
Saved 252 prediction files for text_aware_teacher


Saving predictions: student_ce_only:   0%|          | 0/252 [00:00<?, ?it/s]

Copied manifest to Drive: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/mstcn_teacher_student_split1_full/student_ce_only_prediction_manifest.csv
Saved 252 prediction files for student_ce_only


Saving predictions: student_kd_video_only:   0%|          | 0/252 [00:00<?, ?it/s]

Copied manifest to Drive: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/mstcn_teacher_student_split1_full/student_kd_video_only_prediction_manifest.csv
Saved 252 prediction files for student_kd_video_only
Copied combined prediction manifest to Drive: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/mstcn_teacher_student_split1_full/prediction_manifest_all_models.csv
Copied prediction summary to Drive: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/mstcn_teacher_student_split1_full/prediction_summary.json


,model,video_id,num_frames,local_prediction_path,drive_prediction_path,copied_to_drive
0,baseline_visual_only,P03_cam01_P03_cereals,832,/content/text_assisted_tas_runs/breakfast/mstc...,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,True
1,baseline_visual_only,P03_cam01_P03_coffee,917,/content/text_assisted_tas_runs/breakfast/mstc...,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,True
2,baseline_visual_only,P03_cam01_P03_friedegg,4266,/content/text_assisted_tas_runs/breakfast/mstc...,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,True
3,baseline_visual_only,P03_cam01_P03_milk,1158,/content/text_assisted_tas_runs/breakfast/mstc...,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,True
4,baseline_visual_only,P03_cam01_P03_salat,4449,/content/text_assisted_tas_runs/breakfast/mstc...,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,True


Prediction saving complete.


## Check saved prediction folders

This small check verifies that prediction files exist for each model.


In [17]:
for model_name in models_to_save.keys():
    local_pred_dir = LOCAL_RUN_ROOT / "predictions" / model_name
    drive_pred_dir = RUN_ROOT / "predictions" / model_name

    local_count = len(list(local_pred_dir.glob("*.txt")))
    drive_count = len(list(drive_pred_dir.glob("*.txt")))

    print(f"{model_name:25s} local={local_count:4d} drive={drive_count:4d}")

print("Expected test videos:", len(test_ids))


baseline_visual_only      local= 252 drive= 252
text_aware_teacher        local= 252 drive= 252
student_ce_only           local= 252 drive= 252
student_kd_video_only     local= 252 drive= 252
Expected test videos: 252
